# Prerequisites

In [3]:
# Limit number of cores
import os
os.nice(19)
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

# Create logger
import logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

if not logger.hasHandlers():
    ch = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    ch.setFormatter(formatter)
    logger.addHandler(ch)

In [74]:
import pyccl as ccl
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import h5py
from getdist import loadMCSamples, plots

rootpath = '/home/dneup16/leiden_phd/scripts/GNAT'
if rootpath not in os.sys.path:
    os.sys.path.append(rootpath)

import src.galaxy_alignment_prediction_tool.multipoles
import src.galaxy_alignment_prediction_tool.powerspectrum
import src.galaxy_alignment_prediction_tool.projections
import src.galaxy_alignment_prediction_tool.fitter
import src.galaxy_alignment_prediction_tool.utils

# Reload to update changes
import importlib
importlib.reload(src.galaxy_alignment_prediction_tool.multipoles)
importlib.reload(src.galaxy_alignment_prediction_tool.powerspectrum)
importlib.reload(src.galaxy_alignment_prediction_tool.projections)
importlib.reload(src.galaxy_alignment_prediction_tool.fitter)
importlib.reload(src.galaxy_alignment_prediction_tool.utils)

from src.galaxy_alignment_prediction_tool.multipoles import multipoles
from src.galaxy_alignment_prediction_tool.powerspectrum import powerSpectrum
from src.galaxy_alignment_prediction_tool.projections import projections
from src.galaxy_alignment_prediction_tool.fitter import fitter
from src.galaxy_alignment_prediction_tool.utils import ioUtils

# Initialise classes
fitterHandler = fitter(logger=logger)
ioUtilsHandler = ioUtils(logger=logger)
powerSpectrumHandler = powerSpectrum()
multipolesHandler = multipoles()
projectionsHandler = projections()

# Helper functions

In [113]:
def create_contour_comparison_plot(
    snapshot, 
    sim, 
    selection, 
    estimator,
    figpath,
    respath,
    redshift_dict,
    param_names=None,
) -> None:
    filepath_vanilla = f'{respath}run_20260311_bugfix/{selection}/fit_results_{estimator}_{sim}/fit_results_Snapshot_{snapshot}'
    filepath_NL_scaled = f'{respath}run_20260311_NL_scaling/{selection}/fit_results_{estimator}_{sim}/fit_results_Snapshot_{snapshot}'

    samples_vanilla = loadMCSamples(f'{filepath_vanilla}/mcmc')
    samples_nl = loadMCSamples(f'{filepath_NL_scaled}/mcmc')

    if param_names is None:
        param_names = [param.name for param in samples_nl.getParamNames().names]

    g = plots.get_subplot_plotter(
        subplot_size=2.5,
    )
    g.triangle_plot([samples_vanilla, samples_nl], legend_labels=['Vanilla', 'NL Scaled'], params=param_names)
    plt.suptitle(f'Snapshot {snapshot} (z={redshift_dict.get(str(snapshot), "Unknown")}) - {estimator} - {sim} - {selection}', y=1.03)
    plt.savefig(f'{figpath}comparison_mcmc_Snapshot_{selection}_{estimator}_{sim}_{snapshot}.png', bbox_inches='tight')
    plt.close()

def create_redshift_evolution_comparison_plot(
    sim, 
    selection, 
    estimator,
    figpath,
    respath,
):

    catpath_vanilla = f'{respath}run_20260311_bugfix/IA_fitting_results_summary_{selection}.csv'
    catpath_nl_scaled = f'{respath}run_20260311_NL_scaling/IA_fitting_results_summary_{selection}.csv'

    df_vanilla = pd.read_csv(catpath_vanilla, sep='\t')
    df_nl_scaled = pd.read_csv(catpath_nl_scaled, sep='\t')

    df_vanilla_cut = df_vanilla[(df_vanilla['simulation'] == sim) & (df_vanilla['estimator'] == estimator)]
    df_nl_scaled_cut = df_nl_scaled[(df_nl_scaled['simulation'] == sim) & (df_nl_scaled['estimator'] == estimator)]

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].errorbar(df_vanilla_cut['redshift'], df_vanilla_cut['A_IA'], yerr=df_vanilla_cut['A_IA_err'], fmt='o', label='Vanilla')
    axes[0].errorbar(df_nl_scaled_cut['redshift'], df_nl_scaled_cut['A_IA'], yerr=df_nl_scaled_cut['A_IA_err'], fmt='s', label='NL Scaled')
    axes[0].set_xlabel('Redshift')
    axes[0].set_ylabel(r'$A_{\rm IA}$')
    axes[0].legend(loc='upper left')

    axes[1].errorbar(df_vanilla_cut['redshift'], df_vanilla_cut['b_g'], yerr=df_vanilla_cut['b_g_err'], fmt='o', label='Vanilla')
    axes[1].errorbar(df_nl_scaled_cut['redshift'], df_nl_scaled_cut['b_g'], yerr=df_nl_scaled_cut['b_g_err'], fmt='s', label='NL Scaled')
    axes[1].set_xlabel('Redshift')
    axes[1].set_ylabel(r'$b_{\rm g}$')
    axes[1].legend(loc='upper left')

    plt.suptitle(f'Comparison of IA fitting results - {estimator} - {sim} - {selection}')
    outpath = f'{figpath}/comparison_redshift_evolution_{selection}_{estimator}_{sim}.png'
    plt.savefig(outpath, bbox_inches='tight')
    plt.close()

def create_redshift_evolution_of_NL_scalers(
    sim, 
    selection, 
    figpath,
    respath,    
):
    
    catpath_nl_scaled = f'{respath}run_20260311_NL_scaling/IA_fitting_results_summary_{selection}.csv'

    df_nl_scaled = pd.read_csv(catpath_nl_scaled, sep='\t')

    df_nl_scaled_projections = df_nl_scaled[(df_nl_scaled['simulation'] == sim) & (df_nl_scaled['estimator'] == 'projections')]
    df_nl_scaled_multipoles = df_nl_scaled[(df_nl_scaled['simulation'] == sim) & (df_nl_scaled['estimator'] == 'multipoles')]

    fig, axes = plt.subplots(2, 2, figsize=(11, 10))
    axes_flattened = axes.flatten()

    param_list = ['A_IA', 'b_g', 'alpha_NLgg', 'alpha_NLgp']
    label_list = [r'$A_{\rm IA}$', r'$b_{\rm g}$', r'$\alpha^{\rm NL}_{\rm gg}$', r'$\alpha^{\rm NL}_{\rm gp}$']

    for ax_idx in range(4):
        param = param_list[ax_idx]
        label = label_list[ax_idx]
        axes_flattened[ax_idx].errorbar(df_nl_scaled_projections['redshift'], df_nl_scaled_projections[param], yerr=df_nl_scaled_projections[f'{param}_err'], fmt='o', label='Projections')
        axes_flattened[ax_idx].errorbar(df_nl_scaled_multipoles['redshift'], df_nl_scaled_multipoles[param], yerr=df_nl_scaled_multipoles[f'{param}_err'], fmt='s', label='Multipoles')
        axes_flattened[ax_idx].set_xlabel('Redshift')
        axes_flattened[ax_idx].set_ylabel(label)
        axes_flattened[ax_idx].legend(loc='upper left')

    plt.suptitle(f'Comparison of NL scaling - {sim} - {selection}', y=0.91)
    outpath = f'{figpath}/comparison_NL_scaling_{selection}_{sim}.png'
    plt.savefig(outpath, bbox_inches='tight')
    plt.close()

# Save comparison contour plot of MCMCs

In [ ]:
sim = 'L400_m7'
probe = 'DM'
estimator = 'multipoles'

figpath = '/home/dneup16/leiden_phd/scripts/results/IA_redshift_dependency_simulations/comparison_plots/mcmc_results/'
respath = '/home/dneup16/leiden_phd/scripts/results/IA_redshift_dependency_simulations/'

snapshot_list = [68, 76, 84, 92, 102, 127]
estimator_list = ['multipoles', 'projections']
probe_list = ['DM', 'stars']
redshift_dict = {'68': 2.5, '76': 2.0, '84': 1.5, '92': 1.0, '102': 0.5, '127': 0.0}
for probe in probe_list:
    selection = f'{probe}_nstar_gt50_mstar_gt9p27_mDM_gt11p34'
    for estimator in estimator_list:
        for snapshot in snapshot_list:
            create_contour_comparison_plot(
                snapshot=snapshot,
                sim=sim,
                selection=selection,
                estimator=estimator,
                redshift_dict=redshift_dict,
                figpath=figpath,
                respath=respath,
            )

# Plot comparison redshift evolution of different parameters

In [114]:
sim = 'L400_m7'

figpath = '/home/dneup16/leiden_phd/scripts/results/IA_redshift_dependency_simulations/comparison_plots/redshift_evolution/'
figpath_nl_scaling = '/home/dneup16/leiden_phd/scripts/results/IA_redshift_dependency_simulations/comparison_plots/NL_scaling/'
probe_list = ['DM', 'stars']
estimator_list = ['multipoles', 'projections']
for probe in probe_list:
    selection = f'{probe}_nstar_gt50_mstar_gt9p27_mDM_gt11p34'
    for estimator in estimator_list:
        create_redshift_evolution_comparison_plot(
            sim=sim,
            selection=selection,
            estimator=estimator,
            figpath=figpath,
            respath=respath,
        )

    create_redshift_evolution_of_NL_scalers(
        sim=sim,
        selection=selection,
        figpath=figpath_nl_scaling,
        respath=respath,
    )

,simulation,snapshot,redshift,A_IA,A_IA_err,b_g,b_g_err,reduced_chi2,estimator
0,L400_m7,Snapshot_102,0.50,6.888310,0.545072,1.249747,0.085903,0.111965,projections
1,L400_m7,Snapshot_102,0.50,6.301534,0.145375,1.259708,0.024291,4.117261,multipoles
2,L400_m7,Snapshot_127,0.00,4.856878,0.381898,1.087987,0.075810,0.254094,projections
3,L400_m7,Snapshot_127,0.00,4.284979,0.147393,1.084780,0.028600,3.795270,multipoles
4,L400_m7,Snapshot_68,2.50,12.623826,0.972019,2.406967,0.162130,0.412997,projections
5,L400_m7,Snapshot_68,2.50,12.990014,0.261190,2.461328,0.040875,7.267696,multipoles
6,L400_m7,Snapshot_76,2.00,11.347929,0.856771,2.061338,0.134818,0.297121,projections
7,L400_m7,Snapshot_76,2.00,11.467414,0.223764,2.079046,0.034868,2.728432,multipoles
8,L400_m7,Snapshot_84,1.50,10.175135,0.834291,1.738793,0.122926,0.276202,projections
9,L400_m7,Snapshot_84,1.50,9.891032,0.197410,1.756262,0.030174,1.192181,multipoles
